In [ ]:
# Cell 1: Imports & Authentication
import os
import pandas as pd
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from dotenv import load_dotenv
import time

# Load environment variables from the root folder
load_dotenv(dotenv_path='../.env')

client_id = os.getenv('SPOTIPY_CLIENT_ID')
client_secret = os.getenv('SPOTIPY_CLIENT_SECRET')

# Initialize Spotipy client
sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=client_id,
    client_secret=client_secret
))
print("Spotify API authenticated successfully!")

# Cell 2: Define Target Playlists to gather "Not Hot" songs
# We pull from curated generic playlists to get a wide dynamic range of audio features
playlist_ids = [
    '37i9dQZF1DX4XN36IYwRWW',  # Rock Classics
    '37i9dQZF1DX1lVhptvR8gO',  # Late Night Jazz
    '37i9dQZF1DXdLt7X3U5g9F',  # Classical Essentials
    '37i9dQZF1DX6J5NfM26xc7',  # 90s Hip Hop
    '37i9dQZF1DX8a1td6ZgI7G'   # Deep House Relax
]

# Cell 3: Fetch Track IDs and Metadata
track_data = []

for playlist_id in playlist_ids:
    results = sp.playlist_tracks(playlist_id)
    tracks = results['items']
    
    # Handle pagination if playlist has more than 100 tracks
    while results['next']:
        results = sp.next(results)
        tracks.extend(results['items'])
        
    for item in tracks:
        track = item['track']
        if track: # Filter out local files or dead links
            track_data.append({
                'id': track['id'],
                'song': track['name'].lower(),
                'artist': track['artists'][0]['name'].lower()
            })

df_tracks = pd.DataFrame(track_data).drop_duplicates(subset=['id']).reset_index(drop=True)
print(f"Gathered {len(df_tracks)} unique track IDs from Spotify playlists.")

# Cell 4: Batch Extract Audio Features
# Spotify allows batching 50 track IDs at a time for audio features
audio_features_list = []

for i in range(0, len(df_tracks), 50):
    batch_ids = df_tracks['id'].iloc[i:i+50].tolist()
    features = sp.audio_features(batch_ids)
    
    # Filter out None results
    features = [f for f in features if f is not None]
    audio_features_list.extend(features)
    
    # Polite API scraping delay
    time.sleep(0.1)

df_features = pd.DataFrame(audio_features_list)

# Cell 5: Merge Metadata with Audio Features and Save
# Keep only the essential features for modeling
keep_features = ['id', 'danceability', 'energy', 'loudness', 'speechiness', 
                 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
df_features = df_features[keep_features]

# Merge back with track names and artists
df_final_pool = pd.merge(df_tracks, df_features, on='id')

df_final_pool.to_csv('../data/spotify_songs_pool.csv', index=False)
print(f"Successfully saved {len(df_final_pool)} songs with audio features!")